In [ ]:
# !pip install transformers
# !pip install datasets
# !pip install torchaudio
# !pip install evaluate
# !pip install numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 6.1 MB/s eta 0:00:00


In [ ]:
# STAGE 0: Environment & Dependency Setup
import os
import torch
import evaluate
import numpy as np
import warnings
from datasets import Dataset, DatasetDict, Audio
from transformers import (
    AutoFeatureExtractor,
    Wav2Vec2ForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

warnings.filterwarnings("ignore")

try:
    PROJECT_DIR = '/content/drive/MyDrive/DeepFake_Audio_Detection'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print(f"Checkpoints will be safely saved to: {PROJECT_DIR}")
except ImportError:
    PROJECT_DIR = './DeepFake_Audio_Detection'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print(f"Running locally. Checkpoints will be saved to: {PROJECT_DIR}")

# STAGE 1: Robust Dataset Loading (Folder-Specific)

MODEL_ID = "facebook/wav2vec2-base"
SAMPLE_RATE = 16000
MAX_DURATION = 2.0

DATA_DIR = '/content/drive/MyDrive/Datasets/for-2seconds'

if not os.path.exists(DATA_DIR):
    raise FileNotFoundError(f"CRITICAL ERROR: Cannot find the path {DATA_DIR}. Please check if Google Drive is mounted and the spelling/capitalization is exact.")
else:
    print(f"Path confirmed. Searching inside: {DATA_DIR}")

def load_split(split_dir):
    file_paths = []
    labels = []
    label_map = {"real": 0, "fake": 1}

    for label_str, label_idx in label_map.items():
        folder_path = os.path.join(split_dir, label_str)
        if not os.path.exists(folder_path):
            print(f"  -> Warning: Subdirectory not found - {folder_path}")
            continue

        for root, _, files in os.walk(folder_path):
            for file in files:
                if file.lower().endswith(('.wav', '.mp3', '.flac', '.m4a', '.ogg')):
                    full_path = os.path.join(root, file)
                    file_paths.append(full_path)
                    labels.append(label_idx)

    if not file_paths:
        return None

    print(f"  -> Successfully loaded {len(file_paths)} audio files from {os.path.basename(split_dir)}.")
    dataset = Dataset.from_dict({"audio": file_paths, "label": labels})
    dataset = dataset.cast_column("audio", Audio(sampling_rate=SAMPLE_RATE))
    return dataset

split_names = ['training', 'validation', 'testing']
datasets_dict = {}

for split in split_names:
    split_path = os.path.join(DATA_DIR, split)
    print(f"\nScanning '{split}' directory...")

    if os.path.exists(split_path):
        ds = load_split(split_path)
        if ds:
            hf_split_name = 'train' if split == 'training' else ('test' if split == 'testing' else split)
            datasets_dict[hf_split_name] = ds
    else:
        print(f"  -> Skipping. Split directory does not exist: {split_path}")

full_dataset = DatasetDict(datasets_dict)
print("\nDataset Loading Complete! Structure:")
print(full_dataset)

if len(full_dataset.keys()) == 0:
    raise ValueError("CRITICAL ERROR: Dataset is completely empty. The script has halted. Please verify your Google Drive is properly mounted and the folders contain audio files.")


Checkpoints will be safely saved to: /content/drive/MyDrive/DeepFake_Audio_Detection
Path confirmed. Searching inside: /content/drive/MyDrive/Datasets/for-2seconds

Scanning 'training' directory...
  -> Successfully loaded 13956 audio files from training.

Scanning 'validation' directory...
  -> Successfully loaded 2826 audio files from validation.

Scanning 'testing' directory...
  -> Successfully loaded 1088 audio files from testing.

Dataset Loading Complete! Structure:
DatasetDict({
    train: Dataset({
        features: ['audio', 'label'],
        num_rows: 13956
    })
    validation: Dataset({
        features: ['audio', 'label'],
        num_rows: 2826
    })
    test: Dataset({
        features: ['audio', 'label'],
        num_rows: 1088
    })
})


In [ ]:
# STAGE 2: Feature Extraction Pipeline
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

def preprocess_function(examples):
    # Extract audio arrays safely
    audio_arrays = [x["array"] for x in examples["audio"]]

    inputs = feature_extractor(
        audio_arrays,
        sampling_rate=SAMPLE_RATE,
        max_length=int(SAMPLE_RATE * MAX_DURATION),
        truncation=True,
        padding="max_length"
    )
    return inputs

print("\nPreprocessing datasets (this happens lazily during mapping)...")

encoded_datasets = full_dataset.map(
    preprocess_function,
    remove_columns=["audio"],
    batched=True,
    batch_size=100
)

# STAGE 3: Model & Metrics Initialization
id2label = {0: "REAL", 1: "FAKE"}
label2id = {"REAL": 0, "FAKE": 1}

model = Wav2Vec2ForSequenceClassification.from_pretrained(
    MODEL_ID,
    num_labels=2,
    label2id=label2id,
    id2label=id2label
)

model.freeze_feature_encoder()

accuracy_metric = evaluate.load("accuracy")
f1_metric = evaluate.load("f1")

def compute_metrics(eval_pred):
    predictions = np.argmax(eval_pred.predictions, axis=1)
    accuracy = accuracy_metric.compute(predictions=predictions, references=eval_pred.label_ids)
    f1 = f1_metric.compute(predictions=predictions, references=eval_pred.label_ids)
    return {**accuracy, **f1}

preprocessor_config.json:   0%|          | 0.00/159 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]


Preprocessing datasets (this happens lazily during mapping)...


Map:   0%|          | 0/13956 [00:00<?, ? examples/s]

Map:   0%|          | 0/2826 [00:00<?, ? examples/s]

Map:   0%|          | 0/1088 [00:00<?, ? examples/s]

pytorch_model.bin:   0%|          | 0.00/380M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

model.safetensors:   0%|          | 0.00/380M [00:00<?, ?B/s]

Wav2Vec2ForSequenceClassification LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     | 
-----------------------------+------------+-
project_hid.bias             | UNEXPECTED | 
project_hid.weight           | UNEXPECTED | 
quantizer.codevectors        | UNEXPECTED | 
quantizer.weight_proj.bias   | UNEXPECTED | 
project_q.weight             | UNEXPECTED | 
project_q.bias               | UNEXPECTED | 
quantizer.weight_proj.weight | UNEXPECTED | 
classifier.bias              | MISSING    | 
projector.bias               | MISSING    | 
classifier.weight            | MISSING    | 
projector.weight             | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
# STAGE 4: Training & Evaluation
training_args = TrainingArguments(
    output_dir=os.path.join(PROJECT_DIR, "wav2vec2-for-deepfake"),
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=2,
    num_train_epochs=5,
    warmup_steps=400,
    fp16=True,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    save_total_limit=2,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=encoded_datasets["train"],
    eval_dataset=encoded_datasets["validation"],
    processing_class=feature_extractor,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Starting training process...")
trainer.train()

print("Evaluating the best model on the test set...")
eval_results = trainer.evaluate(encoded_datasets["test"])
print(f"Final Evaluation Results: {eval_results}")

# Save the final optimized model to your persistent directory
final_model_path = os.path.join(PROJECT_DIR, "best_deepfake_model")
trainer.save_model(final_model_path)
print(f"Training complete. Best model saved permanently to {final_model_path}")

Starting training process...


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.075567,0.007654,0.998938,0.998939
2,0.015014,0.024407,0.997169,0.997175
3,0.000135,0.008431,0.998938,0.998939


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Evaluating the best model on the test set...


Final Evaluation Results: {'eval_loss': 2.2028422355651855, 'eval_accuracy': 0.671875, 'eval_f1': 0.5116279069767442, 'eval_runtime': 28.8824, 'eval_samples_per_second': 37.67, 'eval_steps_per_second': 4.709, 'epoch': 3.0}


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete. Best model saved permanently to /content/drive/MyDrive/DeepFake_Audio_Detection/best_deepfake_model


In [ ]:
import os
from google.colab import files
import shutil


model_save_path = os.path.join(PROJECT_DIR, "best_deepfake_model")

zip_filename_desired = "best_deepfake_model.zip"

output_zip_base_path = os.path.join("/content/", os.path.splitext(zip_filename_desired)[0])

print(f"Creating zip archive of '{model_save_path}' to '{output_zip_base_path}.zip'")


archive_full_path = shutil.make_archive(
    output_zip_base_path,
    'zip',
    root_dir=PROJECT_DIR,
    base_dir=os.path.basename(model_save_path)
)

print(f"Your model '{archive_full_path}' is ready for download.")
files.download(archive_full_path)


Creating zip archive of '/content/drive/MyDrive/DeepFake_Audio_Detection/best_deepfake_model' to '/content/best_deepfake_model.zip'
Your model '/content/best_deepfake_model.zip' is ready for download.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Driver code for the saved model for deployment

In [ ]:
import os
import torch
import torchaudio
from transformers import AutoFeatureExtractor, Wav2Vec2ForSequenceClassification

audio_file_path = '/content/Recording (6).m4a'
pytorch_model_path = final_model_path

MODEL_ID = "facebook/wav2vec2-base"
SAMPLE_RATE = 16000
MAX_DURATION = 2.0
id2label = {0: "REAL", 1: "FAKE"}

# Load the feature extractor
feature_extractor = AutoFeatureExtractor.from_pretrained(MODEL_ID)

print(f"Audio file to test: {audio_file_path}")
print(f"PyTorch model to load from: {pytorch_model_path}")

In [ ]:
# Load the trained PyTorch model
print("Loading PyTorch model...")
model = Wav2Vec2ForSequenceClassification.from_pretrained(pytorch_model_path)
model.eval() # Set model to evaluation mode

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
print(f"PyTorch model loaded and moved to {device}.")

Loading PyTorch model...


Loading weights:   0%|          | 0/215 [00:00<?, ?it/s]

PyTorch model loaded and moved to cuda.


In [ ]:
# Load and preprocess the raw audio file
print(f"Loading and preprocessing audio file: {audio_file_path}")

if not os.path.exists(audio_file_path):
    raise FileNotFoundError(f"Error: Audio file not found at {audio_file_path}")

sound, sr = torchaudio.load(audio_file_path)

if sr != SAMPLE_RATE:
    resampler = torchaudio.transforms.Resample(orig_freq=sr, new_freq=SAMPLE_RATE)
    sound = resampler(sound)

if sound.shape[0] > 1:
    sound = sound[0].unsqueeze(0)

inputs = feature_extractor(
    sound.squeeze().numpy(),
    sampling_rate=SAMPLE_RATE,
    max_length=int(SAMPLE_RATE * MAX_DURATION),
    truncation=True,
    padding="max_length",
    return_tensors="pt"
)

inputs = {k: v.to(device) for k, v in inputs.items()}

print("Audio preprocessing complete.")

Loading and preprocessing audio file: /content/Recording (6).m4a
Audio preprocessing complete.


In [ ]:
# Make a prediction
print("Making prediction...")
with torch.no_grad():
    logits = model(**inputs).logits

probabilities = torch.softmax(logits, dim=1).cpu().numpy()[0]
predicted_class_id = torch.argmax(logits, dim=1).item()
predicted_class_label = id2label[predicted_class_id]

print("\n--- Prediction Results ---")
print(f"Audio file: {audio_file_path}")
print(f"Predicted Label: {predicted_class_label}")
print(f"Confidence (REAL): {probabilities[0]:.4f}")
print(f"Confidence (FAKE): {probabilities[1]:.4f}")
print("--------------------------")


Making prediction...

--- Prediction Results ---
Audio file: /content/Recording (6).m4a
Predicted Label: FAKE
Confidence (REAL): 0.0365
Confidence (FAKE): 0.9635
--------------------------
